In [1]:
import npc_lims
import polars as pl

In [2]:
units = pl.read_parquet(npc_lims.get_cache_path('units'))

C:\Users\ben.hardcastle\AppData\Local\Temp\ipykernel_28152\4216505945.py:1: UserWarning: '(default_)region' not set; polars will try to get it from bucket

Set the region manually to silence this warning.
  units = pl.read_parquet(npc_lims.get_cache_path('units'))


In [3]:
structures = ['ORBl', 'ACAd', 'MOs', 'AUDp', 'VISp', 'CA1', 'MRN', 'LGd', 'CP', 'SCiw']
units = units.filter(pl.col('structure').is_in(structures))

In [10]:
import numpy as np
import altair as alt

thresholds = tuple(np.linspace(0, 0.5, 20, endpoint=True))
filters = [pl.col('amplitude_cutoff') < 0.1, pl.col('presence_ratio') > 0.9]
# filters = [pl.lit(True)]
results = []
metrics = ['isi_violations_ratio', 'sliding_rp_violation']
for structure, df in units.filter(*filters).group_by('structure'):
    for metric in metrics:
        result = {}
        result['structure'] = structure[0]
        passing = []
        for thr in thresholds:
            passing.append(
                len(df.filter(pl.col(metric) < thr)) / len(df)
            )
        result['passing'] = passing
        result['metric'] = metric
        result['count'] = len(df)
        result['threshold'] = thresholds
        results.append(result)
    
(
    pl.DataFrame(results)
    .explode(['passing', 'threshold'])
    .plot
    .line(
        x='threshold',
        y='passing', 
        color='structure',
        column='metric',
    )
    .properties(
        title={
            "text": 'Fraction of units passing metric thresholds', 
            "subtitle": [str(f) for f in filters],
            "anchor": "start",
            "orient": 'bottom',
        },
    )
    .interactive()
)

alt.Chart(...)

In [4]:
units.group_by('structure').agg(pl.all().median())

structure,amplitude_cutoff,amplitude_cv_median,amplitude_cv_range,amplitude_median,drift_ptp,drift_std,drift_mad,firing_range,firing_rate,isi_violations_ratio,isi_violations_count,num_spikes,presence_ratio,rp_contamination,rp_violations,sliding_rp_violation,snr,sync_spike_2,sync_spike_4,sync_spike_8,d_prime,isolation_distance,l_ratio,silhouette,nn_hit_rate,nn_miss_rate,exp_decay,half_width,num_negative_peaks,num_positive_peaks,peak_to_valley,peak_trough_ratio,recovery_slope,repolarization_slope,spread,velocity_above,velocity_below,electrode_group_name,peak_channel,cluster_id,default_qc,amplitude,unit_id,ccf_ap,ccf_dv,ccf_ml,location,peak_electrode,obs_intervals,device_name,session_idx,date,subject_id,session_id,id
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,str,f64,f64,f64,str,f64,list[list[f64]],str,f64,datetime[ms],f64,str,f64
"""ORBl""",0.000151,0.229013,0.153708,112.31999,13.498779,2.306463,2.436902,3.2,1.087042,0.018904,2.0,7353.0,1.0,0.009711,1.0,0.065,6.94603,0.075803,0.000069,0.0,3.650416,99.037784,0.764449,0.036837,0.242424,0.006009,0.027295,0.000197,1.0,1.0,0.000573,-0.390436,-78332.69839,509542.757946,120.0,344.125645,362.312533,null,121.0,274.0,1.0,161.495125,null,3025.0,3700.0,4250.0,null,526.0,null,null,0.0,2024-06-24 00:00:00,714748.0,null,1113.0
"""ACAd""",0.000184,0.233206,0.179097,100.619995,17.551636,3.120101,3.431946,3.6,0.965489,0.008369,1.0,6494.5,0.991453,0.000987,1.0,0.065,7.087624,0.061792,0.0,0.0,3.557064,101.237667,0.700952,0.036443,0.243478,0.005307,0.027366,0.000203,1.0,1.0,0.000623,-0.384961,-69332.786212,441461.803338,120.0,376.958103,359.32213,null,223.0,360.0,1.0,142.487288,null,4300.0,2100.0,5175.0,null,1351.0,null,null,0.0,2023-11-14 00:00:00,676909.0,null,1801.0
"""CA1""",0.000206,0.290423,0.211132,93.6,16.802612,2.950322,2.884578,2.8,0.869015,0.068883,3.0,5101.0,1.0,0.036639,1.0,0.09,6.659266,0.069232,0.0,0.0,3.477086,96.39845,0.90086,0.048572,0.327511,0.005634,0.024107,0.000197,1.0,1.0,0.000617,-0.315999,-55513.763844,445862.126794,140.0,252.809084,360.004688,null,195.0,259.0,1.0,133.740363,null,8200.0,2375.0,2600.0,null,853.0,null,null,0.0,2023-08-17 00:00:00,668759.0,null,1101.0
"""LGd""",0.000072,0.250083,0.156258,131.04,13.100647,2.267714,2.278133,8.4,3.342356,0.030171,9.0,21259.5,1.0,0.015988,3.0,0.035,5.966326,0.076032,0.000063,0.0,3.591764,94.473868,0.978323,0.023104,0.170338,0.007902,0.030608,0.0002,1.0,1.0,0.00041,-0.531836,-123531.953257,771687.011147,100.0,537.079489,531.118474,null,89.0,230.0,1.0,204.143948,null,7850.0,3350.0,3350.0,null,1294.0,null,null,0.0,2023-10-24 00:00:00,686740.0,null,1799.0
"""AUDp""",0.00022,0.236111,0.166034,93.6,16.039856,2.725577,3.001897,3.0,0.896982,0.021244,1.0,5654.0,1.0,0.006658,1.0,0.075,7.1708163,0.05253,0.000026,0.0,3.608751,100.203526,0.743513,0.043407,0.319587,0.005794,0.028307,0.0002,1.0,1.0,0.000597,-0.38773,-65112.78803,442578.86236,120.0,363.248845,360.1275,null,171.0,249.5,1.0,133.873748,null,8050.0,2900.0,1600.0,null,939.0,null,null,0.0,2023-11-15 00:00:00,681532.0,null,1266.0
"""CP""",0.000395,0.235971,0.193984,91.259995,11.565987,1.980904,1.97083,1.4,0.30492,0.0,0.0,1919.0,0.87931,0.0,0.0,0.08,6.9958153,0.055087,0.0,0.0,3.952009,124.303416,0.216638,0.057675,0.428571,0.004713,0.029093,0.000207,1.0,1.0,0.000627,-0.332695,-50914.068146,395152.211265,120.0,295.420946,263.13747,null,109.0,273.0,0.0,130.225651,null,4875.0,3800.0,3750.0,null,1169.0,null,null,0.0,2023-10-24 00:00:00,681532.0,null,1520.0
"""MRN""",0.000093,0.232418,0.13509,105.299995,12.589386,2.287637,2.045895,4.4,1.380044,0.006854,3.0,9476.0,0.991379,0.003189,1.0,0.02,5.740181,0.069124,0.00002,0.0,4.429202,142.782182,0.097429,0.059678,0.467673,0.006314,0.028615,0.000147,1.0,1.0,0.00026,-0.461928,-75341.185574,946230.085478,120.0,561.726768,599.320769,null,65.0,72.0,1.0,169.462833,null,8825.0,3750.0,4475.0,null,479.0,null,null,0.0,2024-04-18 00:00

In [30]:
alt.data_transformers.enable("vegafusion")

(
    units
    .filter(
        pl.col('isi_violations_ratio') < .1,
    )
    .plot.scatter(
        x='firing_rate',
        y='isi_violations_ratio',
        # color='structure',
        size=alt.value(.5),
        # tooltip=['unit_id', 'structure', 'firing_rate', 'amplitude_cutoff', 'presence_ratio']
    )
)

alt.Chart(...)

In [51]:
alt.data_transformers.enable("vegafusion")

(
    units
    .filter(
        pl.col('isi_violations_ratio') < 1,
    )
    .plot.area(
        # x='firing_rate',
        x=alt.X('isi_violations_ratio').bin(), 
        y='count()',
        # color='structure',
        size=alt.value(.5),
        # tooltip=['unit_id', 'structure', 'firing_rate', 'amplitude_cutoff', 'presence_ratio']
    )
)

alt.Chart(...)

In [ ]:
def get_duplicates_to_delete(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df
        .with_columns(
            pl.col('path').str.extract(r"/experiment(\d+)_").cast(int).alias('experiment'),
            pl.col('path').str.extract(r"_Record Node (\d+)").cast(int).alias('node'),
        )
        .drop_nulls(['experiment', 'node'])    
        .sort('path')
        .group_by(['session_id', 'probe', 'band', 'experiment'])
        .agg(
            pl.col('probe').len().alias('count'),
            pl.col('path').first().alias('first_path'),
            pl.col('path').last().alias('last_path'),
        )
        .filter(
            pl.col('count') > 1,
        )
        .with_columns(
            pl.when(
                pl.col('probe').is_in(['A', 'B', 'C'])
            ).then(
                pl.col('last_path').alias('to_delete')
            ).otherwise(
               pl.col('first_path').alias('to_delete')
            )
        )
        .sort('session_id')
    )


import upath
def filter_dr_sessions(df):
    # tracked = upath.UPath(
    #     "https://raw.githubusercontent.com/AllenInstitute/npc_lims/refs/heads/main/tracked_sessions.yaml"
    # ).read_text()
    tracked = "636766_20230124"
    def in_tracked(t):
        return t[-1] in tracked
    assert "636766_20230124" in tracked 
    return (
        df
        .with_columns(
            pl.col('session_id').str.split("_").list.slice(1,2).list.join("_").str.replace("-", "", literal=True, n=2).alias('dr_id')
        )
        .apply(in_tracked)
    )

# df.pipe(get_duplicates_to_delete).to_dicts()
df.pipe(filter_dr_sessions)
    